# 10 — Subject-disjoint C-NMC splits

**Purpose:** Freeze fold_2 as test and stratify subjects in folds 0+1 for train/validation.

**Required inputs:** Validated cnmc_manifest.csv from notebook 09.

**Saved outputs:** outputs/classifier/splits/{train,val,test,excluded}.csv, summary.json and leakage_report.json.

## 1. Load central configuration

Set `cnmc_root` once in ignored `config.local.yaml`. Source images are never moved or changed.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "classifier_config.py").is_file():
        sys.path.insert(0, str(candidate)); break
from src.classifier_config import classifier_config, classifier_output
cfg = classifier_config()
out = classifier_output(cfg)
print("Project:", cfg["root"])
print("Config:", cfg["config_path"])
print("C-NMC source:", cfg["cnmc_root"])

## 2. Load source manifest

Only labeled training folds are included; preliminary/final challenge images are excluded.

In [ ]:
import pandas as pd
from src.cnmc_data import inspect_cnmc
source, inspection = inspect_cnmc(cfg)
assert inspection["ready"]
display(source.groupby(["fold", "class_name"]).size().rename("cells").reset_index())

## 3. Create or reuse immutable split

This cell reloads the splitter and displays its file and version so Jupyter cannot keep using the original cached function. fold_2 stays fixed; affected candidate subjects are excluded and audited. Conflicting labels stop splitting. Saved splits are reused.

In [ ]:
import importlib, inspect
import src.cnmc_splits as split_module
importlib.invalidate_caches()
split_module = importlib.reload(split_module)
print("Splitter file:", Path(split_module.__file__).resolve())
print("Splitter implementation version:", getattr(split_module, "SPLIT_IMPLEMENTATION_VERSION", "old/missing"))
if (getattr(split_module, "SPLIT_IMPLEMENTATION_VERSION", None) != 2 or
        "excluded" not in inspect.signature(split_module.validate_splits).parameters):
    raise RuntimeError(f"The source file {split_module.__file__} is outdated. Copy the updated src folder "
                       "and notebook 10 into this project, restart the kernel, and rerun from the first cell.")
create_or_load_splits = split_module.create_or_load_splits
validate_splits = split_module.validate_splits
splits, summary = create_or_load_splits(cfg, source)
display(pd.DataFrame(summary["counts"]).T)
print(summary["protocol"])
print("Excluded candidate cells:", summary.get("excluded_images", 0))
print("Excluded candidate subjects:", summary.get("excluded_subjects", []))
print("Leakage audit:", out / "splits" / "leakage_report.json")
assert not (set(splits["train"].subject_id) & set(splits["val"].subject_id))

## 4. Leakage gate

Subjects, absolute paths and SHA-256 image content must not cross splits. Full source coverage including excluded.csv was validated during creation/loading.

In [ ]:
verified = validate_splits(splits)
assert set(splits["test"].fold) == {"fold_2"}
print("PASS — subject/path/content disjoint; saved manifests point to original images")

## Next

Open **11_EfficientNet_B3_Training.ipynb** for the CUDA smoke test and two-phase training.